# 02 — Repeated heating–regrowth simulations (Figures 3–6, Figs. S2 and S8)

Each chain follows one lineage through 100 cycles. In every cycle the carried-over
cell regrows to *N*<sub>cell</sub> = 10<sup>4</sup> cells; each daughter may shift its
(ln δ, ln p) within the intra-strain variability, subject to the inter-strain
variability (Metropolis–Hastings step); the population is heated, and one survivor
starts the next cycle. 10,000 chains per condition; each chain uses its own
posterior draw, so the uncertainty of the fitted distributions is carried through.

Conditions: no heating, and heating for 2δ, 4δ and 6δ (2-, 4- and 6-log reduction
for the mean strain); start from ATCC 33560 or from the inter-strain distribution.

In [ ]:
import sys, os, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import campyevo
from campyevo import n, QUICK, DATA, RESULTS, FIGURES, TABLES, SIMS, POSTERIOR
from campyevo.plotting import setup_style, savefig
setup_style()
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
warnings.simplefilter("ignore", RuntimeWarning)
print("project root:", ROOT, "| QUICK mode:", QUICK)

In [ ]:
from campyevo.posterior import Posterior
from campyevo.sim import SimSettings, simulate, ATCC33560
from campyevo.analysis import condition_table
from campyevo import plotting as P

# Which posterior: the new CmdStan fit, the original PyStan pickle, or the synthetic test fit.
POSTERIOR_FILE = POSTERIOR / "posterior_cmdstan.npz"
OLD_PYSTAN_PICKLE = DATA / "Samples extracted modfied Weibull multi-level MVN stan.pkl"
USE_ORIGINAL_PICKLE = False       # True = reproduce the published runs

if USE_ORIGINAL_PICKLE:
    post = Posterior.from_pystan_pickle(OLD_PYSTAN_PICKLE); tag = "pystan"
elif POSTERIOR_FILE.exists():
    post = Posterior.load(POSTERIOR_FILE); tag = "cmdstan"
else:
    post = Posterior.load(POSTERIOR / "posterior_synthetic.npz"); tag = "synthetic"
    print("!! using the SYNTHETIC posterior: for checking the code only")
print(post.source, "| draws:", post.n_draws)

N_CHAINS = n(10_000, 400)
N_CYCLES = n(100, 40)
N_CELL = 10 ** 4
SEED = 2026
d_mean = post.delta_mean()
T = {"none": 0.0, "2log": 2 * d_mean, "4log": 4 * d_mean, "6log": 6 * d_mean}
print({k: round(v, 3) for k, v in T.items()})

In [ ]:
results = {}
for init in ("atcc", "inter"):
    for c, t in T.items():
        st = SimSettings(t_heating=t, n_cycles=N_CYCLES, n_cell=N_CELL,
                         init="fixed" if init == "atcc" else "inter",
                         init_logpara=ATCC33560 if init == "atcc" else None)
        results[(init, c)] = simulate(post, st, n_chains=N_CHAINS, base_seed=SEED,
                                      cache=SIMS / f"{tag}_main_{init}_{c}_{N_CHAINS}.npz")
        print(init, c, "done")

## Convergence table (replaces the eyeballed cycle counts in the Results)

In [ ]:
tab = condition_table({f"{i} | {c}": r for (i, c), r in results.items()})
tab.to_csv(TABLES / "main_conditions.csv", index=False)
tab

*cycles to 95 %*: first cycle from which the mean ln δ stays beyond 95 % of its total
shift (plateau = mean of the last 20 cycles). Empty when there is no shift (no heating
from the inter-strain distribution).

## Figures 3–6 and Figs. S2, S8

The figures follow one line of argument, each in microbiological units
(δ = time to a 6-log reduction ÷ 6, in min; p = curve shape, > 1 shoulder, < 1 tail):

| Figure | Message |
|---|---|
| 3 | One lineage, followed cycle by cycle, drifts toward greater heat resistance. |
| 4 | Across all lineages: the stronger the heating, the further they move, but the more lineages are lost. |
| 5 | Seen as survival curves: the same heating at 55 °C kills the evolved lineages less. |
| 6 | In practice: how much a treatment loses its effect, how fast, and how many lineages survive. |

In [ ]:
P.setup_style_v2()
inter = {c: results[("inter", c)] for c in T}
atcc = {c: results[("atcc", c)] for c in T}

### Figure 3 — one lineage, cycle by cycle (start: ATCC 33560; the example is the lineage closest to the median end point)

In [ ]:
fig = P.fig3_example_lineages(atcc, post)
savefig(fig, "Fig3_example_lineage"); plt.show()

### Figure 4 — where the lineages end after 100 cycles (start: present-day strains)

In [ ]:
fig = P.fig4_where_lineages_end(inter, post)
savefig(fig, "Fig4_where_lineages_end"); plt.show()

### Figure 5 — survival curves before and after 100 cycles

In [ ]:
fig = P.fig5_survival_curves_before_after(inter, post, T)
savefig(fig, "Fig5_survival_curves"); plt.show()

### Figure 6 — effect of the same treatment over cycles, and lineages surviving

In [ ]:
fig = P.fig6_treatment_effect(inter, atcc, T)
savefig(fig, "Fig6_treatment_effect"); plt.show()

### Numbers for the Results text

log10 reduction achieved by each cycling treatment on the lineages still alive (median over
lineages), and the share of lineages not lost.

In [ ]:
from campyevo.analysis import reduction_over_cycles
rows = []
for (init, c), r in results.items():
    if c == "none":
        continue
    red = reduction_over_cycles(r, T[c]).set_index("cycle")["q50"]
    rows.append({"start": init, "heating": c, "heating time (min)": T[c],
                 **{f"reduction at cycle {k}": red.get(k, np.nan) for k in (0, 10, 20, 50, N_CYCLES)},
                 "lineages not lost (%)": 100 * r.survival_ratio()[-1]})
tab_eff = pd.DataFrame(rows)
tab_eff.to_csv(TABLES / "main_treatment_effect.csv", index=False)
tab_eff

### Fig. S2 — mean δ and p over cycles (former Fig. 5)

In [ ]:
fig = P.figS2_means_over_cycles(inter, atcc)
savefig(fig, "FigS2_means_over_cycles"); plt.show()

### Fig. S8 — Figures 4 and 5 for lineages started from ATCC 33560

In [ ]:
fig = P.fig4_where_lineages_end(atcc, post)
savefig(fig, "FigS8a_where_lineages_end_ATCC33560"); plt.show()
fig = P.fig5_survival_curves_before_after(atcc, post, T, reference=ATCC33560)
savefig(fig, "FigS8b_survival_curves_ATCC33560"); plt.show()